# 🎨 Visualization Gallery

CircuitKit provides multiple visualization modes for inspecting discovered circuits. This notebook demonstrates each one using GPT-2/IOI on CPU.

**Visualization stack:**

| Class | What it shows |
|-------|---------------|
| `CircuitGraphVisualizer` | Interactive node/edge graph (Plotly) |
| `ComparisonDashboard` | Cross-condition heatmaps, correlations |
| `JupyterWidgetSuite` | One-call combined analysis widget |
| `Circuit.plot()` | Convenience wrapper for graph viz |
| `ck.visualize_circuit()` | Flat API dispatcher |

- **Model:** GPT-2 (CPU-friendly)
- **Runtime:** ~5 min

---

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import HTML, display

import circuitkit as ck
from circuitkit import Pipeline

## Create Circuits for Visualization

We'll discover two circuits with different seeds to enable comparison views.

In [ ]:
# Circuit 1: seed 42
pipe1 = Pipeline("gpt2", task="ioi", precision="float32", output_dir="./results/viz/seed42")
pipe1.discover(algorithm="eap-ig", n_examples=32, batch_size=4, ig_steps=5)

# Circuit 2: seed 123 (re-run with a different data sample)
pipe2 = Pipeline("gpt2", task="ioi", precision="float32", output_dir="./results/viz/seed123")
pipe2.discover(algorithm="eap-ig", n_examples=32, batch_size=4, ig_steps=5)

print(f"Circuit 1: {pipe1._circuit}")
print(f"Circuit 2: {pipe2._circuit}")

## Section A: Circuit Graph Visualization

The default visualization. Nodes are sized and colored by importance score. Hover for details.

In [ ]:
# Method 1: via Circuit.plot() — the simplest path
html_default = pipe1._circuit.plot("./results/viz/graph_default.html")
print("Saved: ./results/viz/graph_default.html")
display(HTML(html_default))

In [ ]:
# Method 2: via Pipeline.visualize()
html_pipeline = pipe1.visualize(mode="graph", output="./results/viz/graph_pipeline.html")
print("Saved: ./results/viz/graph_pipeline.html")
display(HTML(html_pipeline))

In [ ]:
# Method 3: via the flat API
html_flat = ck.visualize_circuit(pipe1._circuit, mode="graph", output="./results/viz/graph_flat.html")
print("Saved: ./results/viz/graph_flat.html")
display(HTML(html_flat))

### Direct Graph Visualizer Access

For full control, use `CircuitGraphVisualizer` directly. This gives access to threshold filtering, top-k node extraction, and JSON export.

In [ ]:
from circuitkit.visualize.graph_viz import CircuitGraphVisualizer
from circuitkit.artifacts.scores import CircuitScores

# Build CircuitScores from the circuit
cs = CircuitScores(
    task=pipe1._circuit.task or "ioi",
    model=pipe1._circuit.model_name or "gpt2",
    algorithm=pipe1._circuit.algorithm or "eap-ig",
    level="node",
    node_scores=pipe1._circuit.scores,
    timestamp=CircuitScores.create_timestamp(),
)

graph_dict = {"nodes": {name: {} for name in pipe1._circuit.scores}, "edges": []}
viz = CircuitGraphVisualizer(graph_dict, cs)

# Top-k nodes
top_nodes = viz.get_top_k_nodes(10)
print("Top 10 nodes:")
for name, score in top_nodes.items():
    print(f"  {name:>10s}  {score:.4f}")

In [ ]:
# Export graph data as JSON (for external tools)
viz.export_graph_data("./results/viz/graph_data.json")
print("Exported graph data to ./results/viz/graph_data.json")

## Section B: Comparison Dashboard

Compare two (or more) circuits across multiple dimensions. Takes `Dict[str, Dict[str, float]]` — condition name to node score dict.

In [ ]:
from circuitkit.visualize.comparison import ComparisonDashboard

dashboard = ComparisonDashboard(
    circuits={
        "Seed 42": pipe1._circuit.scores,
        "Seed 123": pipe2._circuit.scores,
    },
    comparison_type="stability",
)

In [ ]:
# Stability heatmap — are the same nodes important across seeds?
fig = dashboard.plot_stability_heatmap(top_k=15)
fig.show()

In [ ]:
# Correlation matrix — how correlated are the two circuits' scores?
fig = dashboard.plot_correlation_matrix()
fig.show()

In [ ]:
# Score distribution boxplot
fig = dashboard.plot_distribution_comparison()
fig.show()

In [ ]:
# Robustness comparison (top-k bar chart)
fig = dashboard.plot_robustness_comparison(top_k=10)
fig.show()

In [ ]:
# Export entire dashboard as HTML
dashboard.export_to_html("./results/viz/comparison_dashboard.html")
print("Saved: ./results/viz/comparison_dashboard.html")
display(HTML(filename="./results/viz/comparison_dashboard.html"))

## Section C: Flat API — `ck.visualize_circuit(mode="comparison")`

The flat API wraps ComparisonDashboard for the common two-circuit case.

In [ ]:
# Compare two circuits via the flat API
result = ck.visualize_circuit(
    pipe1._circuit,
    mode="comparison",
    second_circuit=pipe2._circuit,
    output="./results/viz/comparison_flat.html",
)
print("Saved: ./results/viz/comparison_flat.html")
display(HTML(filename="./results/viz/comparison_flat.html"))

## Section D: Scores Inspection

Sometimes a table is all you need. Here's how to work with raw scores.

In [ ]:
import pandas as pd

# Build a DataFrame from scores
scores_df = pd.DataFrame([
    {"node": name, "score": score}
    for name, score in pipe1._circuit.scores.items()
]).sort_values("score", ascending=False)

print(f"Total nodes scored: {len(scores_df)}")
scores_df.head(15)

In [ ]:
# Score distribution histogram
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores_df["score"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Importance Score")
ax.set_ylabel("Count")
ax.set_title("Node Score Distribution (GPT-2 / IOI / EAP-IG)")
plt.tight_layout()
plt.show()

## Summary

| Method | Use case |
|--------|----------|
| `circuit.plot()` | Quick single-circuit graph |
| `pipe.visualize()` | Same, from a Pipeline |
| `ck.visualize_circuit(mode="graph")` | Same, from the flat API |
| `CircuitGraphVisualizer` | Full control: filter, export JSON |
| `ComparisonDashboard` | Multi-circuit heatmaps + correlations |
| `ck.visualize_circuit(mode="comparison")` | Quick two-circuit comparison |
| `ck.visualize_circuit(mode="dashboard")` | Launch Streamlit app (if installed) |